# Data Collection

In [1]:
import sqlite3
import pandas as pd

## Define Connection to SQLite

In [2]:
conn = sqlite3.connect('../dataset/nfts.sqlite/nfts.sqlite')

## Define Time Range

In [5]:
JUNE_1 = 1622505600   # June 1, 2021 00:00:00 UTC
SEPT_1 = 1630454400   # September 1, 2021 00:00:00 UTC

print(f"\nTime range:")
print(f"From : {pd.to_datetime(JUNE_1, unit='s')}")
print(f"Until: {pd.to_datetime(SEPT_1, unit='s')}")


Time range:
From : 2021-06-01 00:00:00
Until: 2021-09-01 00:00:00


## Load Data 

In [8]:
transfers = pd.read_sql_query(f"""
    SELECT
        t.transaction_hash,
        t.block_number,
        t.timestamp,
        t.nft_address,
        t.token_id,
        t.from_address,
        t.to_address,
        t.transaction_value,
        m.timestamp          AS mint_timestamp,
        sf.transfers_out     AS transfers_out_from,
        sf.transfers_in      AS transfers_in_from,
        st.transfers_out     AS transfers_out_to,
        st.transfers_in      AS transfers_in_to,
        ot.num_transitions
    FROM transfers t
    LEFT JOIN mints m
        ON  t.token_id    = m.token_id
        AND t.nft_address = m.nft_address
    LEFT JOIN transfer_statistics_by_address sf
        ON t.from_address = sf.address
    LEFT JOIN transfer_statistics_by_address st
        ON t.to_address   = st.address
    LEFT JOIN ownership_transitions ot
        ON  t.from_address = ot.from_address
        AND t.to_address   = ot.to_address
    WHERE t.timestamp >= {JUNE_1}
      AND t.timestamp <  {SEPT_1}
""", conn)

print(f"Loaded: {len(transfers):,} rows")
print(f"Sample num_transitions: {transfers['num_transitions'].head()}")

Loaded: 2,539,120 rows
Sample num_transitions: 0    10
1     6
2    19
3     4
4    10
Name: num_transitions, dtype: int64


## Fix Data Types

In [9]:
# Convert timestamps to datetime
transfers['timestamp_dt'] = pd.to_datetime(
    transfers['timestamp'], unit='s'
)
transfers['mint_timestamp_dt'] = pd.to_datetime(
    transfers['mint_timestamp'], unit='s', errors='coerce'
)

# Convert transaction_value to float
transfers['transaction_value'] = pd.to_numeric(
    transfers['transaction_value'], errors='coerce'
).fillna(0.0)

## Distribution per Month

In [11]:
print("\n=== DISTRIBUTION PER MONTH ===")
monthly = transfers.groupby(
    transfers['timestamp_dt'].dt.to_period('M')
).size()
print(monthly)


=== DISTRIBUTION PER MONTH ===
timestamp_dt
2021-06     378995
2021-07     598847
2021-08    1561278
Freq: M, dtype: int64


## Data Filtering

In [14]:
# Aligned with Liu et al. (2023):
# Wash trading only occurs in paid transfers

print("\n=== FILTER 1: Sales Only (value > 0) ===")
before = len(transfers)
sales  = transfers[transfers['transaction_value'] > 0].copy()
after  = len(sales)
print(f"Before : {before:,}")
print(f"After  : {after:,}")
print(f"Removed: {before - after:,} zero-value transfers "
      f"({(before-after)/before:.1%})")


=== FILTER 1: Sales Only (value > 0) ===
Before : 2,539,120
After  : 1,582,618
Removed: 956,502 zero-value transfers (37.7%)


In [16]:
# Burn addresses represent NFT destruction,
# not ownership transfer → irrelevant to wash trading

print("\n=== FILTER 2: Remove Burn Addresses ===")

BURN_ADDRESSES = {
    '0x0000000000000000000000000000000000000000',
    '0x000000000000000000000000000000000000dead'
}
burn_lower = {a.lower() for a in BURN_ADDRESSES}

before      = len(sales)
sales_clean = sales[
    ~sales['from_address'].str.lower().isin(burn_lower) &
    ~sales['to_address'].str.lower().isin(burn_lower)
].copy().reset_index(drop=True)
after = len(sales_clean)

print(f"Before : {before:,}")
print(f"After  : {after:,}")
print(f"Removed: {before - after:,} burn address transactions "
      f"({(before-after)/before:.1%})")


=== FILTER 2: Remove Burn Addresses ===
Before : 1,582,618
After  : 1,563,989
Removed: 18,629 burn address transactions (1.2%)


## Sanity Check

In [17]:
print("\n=== SANITY CHECK ===")
print(f"Columns       : {sales_clean.columns.tolist()}")
print(f"Dtypes:\n{sales_clean.dtypes}")
print(f"Missing values:\n{sales_clean.isnull().sum()}")
print(f"\ntoken_id sample: {sales_clean['token_id'].head(3).tolist()}")
print(f"token_id dtype : {sales_clean['token_id'].dtype}")


=== SANITY CHECK ===
Columns       : ['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'timestamp_dt', 'mint_timestamp_dt']
Dtypes:
transaction_hash                str
block_number                  int64
timestamp                     int64
nft_address                     str
token_id                        str
from_address                    str
to_address                      str
transaction_value           float64
mint_timestamp              float64
transfers_out_from            int64
transfers_in_from             int64
transfers_out_to              int64
transfers_in_to               int64
num_transitions               int64
timestamp_dt          datetime64[s]
mint_timestamp_dt     datetime64[s]
dtype: object
Missing values:
transaction_hash           0
block_number               0
timest

## Temporal Split

In [18]:
print("\n=== TEMPORAL SPLIT 70/30 ===")

sales_clean = sales_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

split_idx  = int(len(sales_clean) * 0.70)
split_time = sales_clean.loc[split_idx, 'timestamp_dt']

train = sales_clean.iloc[:split_idx].copy()
test  = sales_clean.iloc[split_idx:].copy()

print(f"Split point: {split_time}")
print(f"\nTrain: {len(train):,} rows ({len(train)/len(sales_clean):.1%})")
print(f"  From: {train['timestamp_dt'].min()}")
print(f"  To  : {train['timestamp_dt'].max()}")
print(f"\nTest : {len(test):,} rows ({len(test)/len(sales_clean):.1%})")
print(f"  From: {test['timestamp_dt'].min()}")
print(f"  To  : {test['timestamp_dt'].max()}")


=== TEMPORAL SPLIT 70/30 ===
Split point: 2021-08-23 02:03:27

Train: 1,094,792 rows (70.0%)
  From: 2021-06-01 00:00:26
  To  : 2021-08-23 02:03:24

Test : 469,197 rows (30.0%)
  From: 2021-08-23 02:03:27
  To  : 2021-08-31 23:59:55


## Summary

In [20]:
print(f"\n{'='*55}")
print(f"H1 — DATA COLLECTION SUMMARY")
print(f"{'='*55}")
print(f"Source         : Simiotic Ethereum NFT (Kaggle)")
print(f"Period         : June – August 2021")
print(f"Total rows     : {len(sales_clean):,}")
print(f"Unique wallets : "
      f"{pd.concat([sales_clean['from_address'], sales_clean['to_address']]).nunique():,}")
print(f"Unique NFTs    : {sales_clean['nft_address'].nunique():,}")
print(f"Date range     : {sales_clean['timestamp_dt'].min()} "
      f"to {sales_clean['timestamp_dt'].max()}")
print(f"Missing mint   : {sales_clean['mint_timestamp'].isnull().sum():,} "
      f"({sales_clean['mint_timestamp'].isnull().mean():.1%})")
print(f"Train size     : {len(train):,}")
print(f"Test size      : {len(test):,}")


H1 — DATA COLLECTION SUMMARY
Source         : Simiotic Ethereum NFT (Kaggle)
Period         : June – August 2021
Total rows     : 1,563,989
Unique wallets : 202,631
Unique NFTs    : 2,378
Date range     : 2021-06-01 00:00:26 to 2021-08-31 23:59:55
Missing mint   : 110,966 (7.1%)
Train size     : 1,094,792
Test size      : 469,197


In [21]:
output_path = '../dataset/nft_transfers_jun_aug.csv'
sales_clean.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")


Saved to: ../dataset/nft_transfers_jun_aug.csv


# Labeling

In [22]:
sales_clean = sales_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

sales_clean['is_wash_trading'] = 0
wash_hashes = set()

# Constants
MAX_SECS_BUYBACK   = 30 * 24 * 3600  # Rule 1: 30 days
MAX_SECS_MULTIHOP  = 24 * 3600        # Rule 2: 24 hours
MAX_HOPS           = 5                # Rule 2: max chain length

grouped      = sales_clean.groupby(['nft_address', 'token_id'])
total_groups = len(grouped)

print(f"Labeling rules:")
print(f"  Rule 1 (Liu et al. 2023)        : "
      f"Seller buyback < 30 days")
print(f"  Rule 2 (Von Wachter et al. 2022): "
      f"Multi-hop cycle A->...->A < 24 hours "
      f"(max {MAX_HOPS} hops)")
print(f"\nUnique NFT tokens: {total_groups:,}")
print(f"Processing...")

rule1_count = 0
rule2_count = 0
processed   = 0

for (nft_addr, token_id), group in grouped:

    if len(group) < 2:
        processed += 1
        continue

    group = group.sort_values(
        'timestamp'
    ).reset_index(drop=True)
    n = len(group)

    for i in range(n):
        from_i = group.loc[i, 'from_address'].lower()
        to_i   = group.loc[i, 'to_address'].lower()
        hash_i = group.loc[i, 'transaction_hash']
        ts_i   = group.loc[i, 'timestamp']

        # Collect hashes in current chain for Rule 2
        chain_hashes = [hash_i]
        chain_end    = to_i  # last wallet in chain

        for j in range(i + 1, n):
            ts_j      = group.loc[j, 'timestamp']
            diff_secs = ts_j - ts_i

            # ------------------------------------------
            # RULE 1: Seller Buyback (Liu et al., 2023)
            # A sells NFT, then A buys it back
            # within 30 days — regardless of path
            # ------------------------------------------
            if diff_secs > MAX_SECS_BUYBACK:
                break

            from_j = group.loc[j, 'from_address'].lower()
            to_j   = group.loc[j, 'to_address'].lower()
            hash_j = group.loc[j, 'transaction_hash']

            if from_i == to_j:
                wash_hashes.add(hash_i)
                wash_hashes.add(hash_j)
                rule1_count += 1

            # ------------------------------------------
            # RULE 2: Multi-hop Cycle (Von Wachter, 2022)
            # NFT travels A->B->C->...->A within 24 hours
            # Chain must be continuous:
            # each tx's seller = previous tx's buyer
            # ------------------------------------------
            if diff_secs <= MAX_SECS_MULTIHOP:

                # Continue chain if this tx starts
                # where last chain tx ended
                if (from_j == chain_end and
                        len(chain_hashes) < MAX_HOPS):

                    chain_hashes.append(hash_j)
                    chain_end = to_j

                    # Check if cycle is complete:
                    # NFT returned to original sender
                    if to_j == from_i:
                        # Found multi-hop cycle!
                        for h in chain_hashes:
                            wash_hashes.add(h)
                        rule2_count += 1
                        # Reset chain to look for more cycles
                        chain_hashes = [hash_i]
                        chain_end    = to_i

    processed += 1
    if processed % 50000 == 0:
        print(f"  Progress: {processed:,} / {total_groups:,} "
              f"({processed/total_groups:.1%})")

# Apply labels
sales_clean.loc[
    sales_clean['transaction_hash'].isin(wash_hashes),
    'is_wash_trading'
] = 1

Labeling rules:
  Rule 1 (Liu et al. 2023)        : Seller buyback < 30 days
  Rule 2 (Von Wachter et al. 2022): Multi-hop cycle A->...->A < 24 hours (max 5 hops)

Unique NFT tokens: 1,164,974
Processing...
  Progress: 200,000 / 1,164,974 (17.2%)
  Progress: 950,000 / 1,164,974 (81.5%)


In [23]:
total      = len(sales_clean)
wt_count   = int(sales_clean['is_wash_trading'].sum())
norm_count = total - wt_count
ratio      = wt_count / total

print(f"\n{'='*55}")
print(f"H2 — LABELING RESULTS")
print(f"{'='*55}")
print(f"Total sales      : {total:,}")
print(f"Wash trading (1) : {wt_count:,} ({ratio:.3%})")
print(f"Normal (0)       : {norm_count:,} ({1-ratio:.3%})")
print(f"Class imbalance  : 1 : {norm_count//wt_count if wt_count > 0 else 'N/A'}")
print(f"\nBreakdown by rule:")
print(f"  Rule 1 (seller buyback)  : {rule1_count:,} pairs flagged")
print(f"  Rule 2 (multi-hop cycle) : {rule2_count:,} cycles flagged")
print(f"  Overlap (both rules)     : "
      f"{rule1_count + rule2_count*MAX_HOPS - wt_count:,} "
      f"(expected — some transactions caught by both)")

# Sanity check
wt_df      = sales_clean[sales_clean['is_wash_trading'] == 1]
burn_in_wt = wt_df[
    wt_df['to_address'].str.lower().isin(burn_lower)
]
print(f"\nSanity check:")
print(f"  Burn addresses in wash labels: {len(burn_in_wt)}")
print(f"  (Expected: 0)")

# Sample
print(f"\nSample wash trading transactions:")
print(wt_df[['timestamp_dt', 'from_address',
             'to_address', 'transaction_value',
             'is_wash_trading']].head(5).to_string())


H2 — LABELING RESULTS
Total sales      : 1,563,989
Wash trading (1) : 12,420 (0.794%)
Normal (0)       : 1,551,569 (99.206%)
Class imbalance  : 1 : 124

Breakdown by rule:
  Rule 1 (seller buyback)  : 14,571 pairs flagged
  Rule 2 (multi-hop cycle) : 4,811 cycles flagged
  Overlap (both rules)     : 26,206 (expected — some transactions caught by both)

Sanity check:
  Burn addresses in wash labels: 0
  (Expected: 0)

Sample wash trading transactions:
           timestamp_dt                                from_address                                  to_address  transaction_value  is_wash_trading
2   2021-06-01 00:00:30  0x6958F5e95332D93D21af0D7B9Ca85B8212fEE0A5  0x79066aE1d08c5B32F1EE9a42597f1e8F0eb6e23b       4.920000e+13                1
3   2021-06-01 00:00:30  0x8c2a1A80c23a8E9C06D624505B62DA9fa4d53bAa  0x6958F5e95332D93D21af0D7B9Ca85B8212fEE0A5       4.920000e+13                1
53  2021-06-01 00:21:01  0x6958F5e95332D93D21af0D7B9Ca85B8212fEE0A5  0x79066aE1d08c5B32F1EE9a42597f1

In [25]:
output_path = '../dataset/nft_transfers_jun_aug_labeled.csv'
sales_clean.to_csv(output_path, index=False)
print(f"\nUpdated CSV with labels: {output_path}")


Updated CSV with labels: ../dataset/nft_transfers_jun_aug_labeled.csv


# Feature Engineering

In [6]:
import numpy as np
import pandas as pd

# Load labeled dataset
print("Loading labeled dataset...")
sales_clean = pd.read_csv('../dataset/nft_transfers_jun_aug_labeled.csv')

# Fix timestamp columns (read as string from CSV)
sales_clean['timestamp_dt'] = pd.to_datetime(
    sales_clean['timestamp_dt']
)
sales_clean['mint_timestamp_dt'] = pd.to_datetime(
    sales_clean['mint_timestamp_dt'], errors='coerce'
)

print(f"Loaded: {len(sales_clean):,} rows")
print(f"Columns: {sales_clean.columns.tolist()}")
print(f"Wash trading: {sales_clean['is_wash_trading'].sum():,} "
      f"({sales_clean['is_wash_trading'].mean():.3%})")

Loading labeled dataset...
Loaded: 1,563,989 rows
Columns: ['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'timestamp_dt', 'mint_timestamp_dt', 'is_wash_trading']
Wash trading: 12,420 (0.794%)


In [7]:
# ============================================================
# FEATURE 1: Holding Time
# Source: Von Wachter et al. (2022), Liu et al. (2023)
# Definition: How long the seller held the NFT before selling
# = timestamp of sale - mint_timestamp
# Unit: hours
# ============================================================
print("\nComputing Feature 1: holding_time_hours...")

sales_clean['holding_time_hours'] = (
    sales_clean['timestamp'] - sales_clean['mint_timestamp']
) / 3600  # convert seconds to hours

# Handle missing mint_timestamp (9% missing)
# Fill with median holding time — conservative approach
median_holding = sales_clean['holding_time_hours'].median()
sales_clean['holding_time_hours'] = sales_clean[
    'holding_time_hours'
].fillna(median_holding)

# Cap negative values (mint after sale — data inconsistency)
sales_clean['holding_time_hours'] = sales_clean[
    'holding_time_hours'
].clip(lower=0)

print(f"  Median holding time : {median_holding:.2f} hours")
print(f"  Min holding time    : {sales_clean['holding_time_hours'].min():.2f} hours")
print(f"  Max holding time    : {sales_clean['holding_time_hours'].max():.2f} hours")
print(f"  Missing filled      : {sales_clean['holding_time_hours'].isna().sum()}")



Computing Feature 1: holding_time_hours...
  Median holding time : 69.80 hours
  Min holding time    : 0.00 hours
  Max holding time    : 3646.02 hours
  Missing filled      : 0


In [8]:
# ============================================================
# FEATURE 2: Pair Frequency
# Source: La Morgia et al. (2023), NFTGraph (2021)
# Definition: How many times this exact (from, to) wallet pair
# has traded the same NFT token
# ============================================================
print("\nComputing Feature 2: pair_frequency...")

pair_freq = sales_clean.groupby(
    ['nft_address', 'token_id', 'from_address', 'to_address']
).size().reset_index(name='pair_frequency')

sales_clean = sales_clean.merge(
    pair_freq,
    on=['nft_address', 'token_id', 'from_address', 'to_address'],
    how='left'
)

print(f"  Mean pair frequency : {sales_clean['pair_frequency'].mean():.3f}")
print(f"  Max pair frequency  : {sales_clean['pair_frequency'].max()}")
print(f"  Pairs with freq > 1 : {(sales_clean['pair_frequency'] > 1).sum():,}")



Computing Feature 2: pair_frequency...
  Mean pair frequency : 1.010
  Max pair frequency  : 25
  Pairs with freq > 1 : 2,864


In [9]:
# ============================================================
# FEATURE 3: Time Since Last Trade (on same NFT)
# Source: Liu et al. (2023) — time component heuristics
# Definition: Hours since the previous transaction
# on the same NFT token
# ============================================================
print("\nComputing Feature 3: time_since_last_hours...")

# Sort by nft + token + timestamp
sales_clean = sales_clean.sort_values(
    ['nft_address', 'token_id', 'timestamp']
).reset_index(drop=True)

# Shift timestamp within each NFT group
sales_clean['prev_timestamp'] = sales_clean.groupby(
    ['nft_address', 'token_id']
)['timestamp'].shift(1)

sales_clean['time_since_last_hours'] = (
    sales_clean['timestamp'] - sales_clean['prev_timestamp']
) / 3600

# First transaction of each NFT has no previous → fill with median
median_interval = sales_clean['time_since_last_hours'].median()
sales_clean['time_since_last_hours'] = sales_clean[
    'time_since_last_hours'
].fillna(median_interval)

# Drop helper column
sales_clean = sales_clean.drop(columns=['prev_timestamp'])

print(f"  Median interval     : {median_interval:.2f} hours")
print(f"  Min interval        : {sales_clean['time_since_last_hours'].min():.2f} hours")
print(f"  Max interval        : {sales_clean['time_since_last_hours'].max():.2f} hours")



Computing Feature 3: time_since_last_hours...
  Median interval     : 97.19 hours
  Min interval        : 0.00 hours
  Max interval        : 2181.26 hours


In [10]:
# ============================================================
# FEATURE 4 & 5: Symmetry Ratio
# Source: Von Wachter et al. (2022)
# Definition: transfers_in / transfers_out per wallet
# Wash traders have balanced in/out (ratio ≈ 1.0)
# Add small epsilon to avoid division by zero
# ============================================================
print("\nComputing Feature 4 & 5: symmetry_ratio...")

EPSILON = 1e-6

sales_clean['symmetry_ratio_from'] = (
    sales_clean['transfers_in_from'] /
    (sales_clean['transfers_out_from'] + EPSILON)
)

sales_clean['symmetry_ratio_to'] = (
    sales_clean['transfers_in_to'] /
    (sales_clean['transfers_out_to'] + EPSILON)
)

print(f"  symmetry_ratio_from mean: {sales_clean['symmetry_ratio_from'].mean():.3f}")
print(f"  symmetry_ratio_to mean  : {sales_clean['symmetry_ratio_to'].mean():.3f}")



Computing Feature 4 & 5: symmetry_ratio...
  symmetry_ratio_from mean: 0.697
  symmetry_ratio_to mean  : 1922812.619


In [11]:
# ============================================================
# VALIDATION: Compare features between wash trading
# and normal transactions
# ============================================================
print(f"\n{'='*55}")
print(f"FEATURE VALIDATION")
print(f"(Wash Trading vs Normal — mean values)")
print(f"{'='*55}")

features_to_check = [
    'holding_time_hours',
    'pair_frequency',
    'time_since_last_hours',
    'symmetry_ratio_from',
    'symmetry_ratio_to'
]

wt     = sales_clean[sales_clean['is_wash_trading'] == 1]
normal = sales_clean[sales_clean['is_wash_trading'] == 0]

print(f"\n{'Feature':<25} {'Wash Trading':>15} {'Normal':>15} {'Ratio':>10}")
print("-" * 65)
for feat in features_to_check:
    wt_mean  = wt[feat].mean()
    nor_mean = normal[feat].mean()
    ratio    = wt_mean / nor_mean if nor_mean != 0 else float('inf')
    print(f"{feat:<25} {wt_mean:>15.3f} {nor_mean:>15.3f} {ratio:>10.2f}x")



FEATURE VALIDATION
(Wash Trading vs Normal — mean values)

Feature                      Wash Trading          Normal      Ratio
-----------------------------------------------------------------
holding_time_hours                338.527         259.691       1.30x
pair_frequency                      2.232           1.001       2.23x
time_since_last_hours              48.754         131.188       0.37x
symmetry_ratio_from                 0.919           0.695       1.32x
symmetry_ratio_to             4474077.714     1902390.251       2.35x


In [13]:
# Fix symmetry_ratio_to — cap outliers
# Ratio > 100 tidak meaningful secara bisnis
# Wallet yang hanya punya 1 transfer_in dan 0 transfer_out
# akan dapat ratio = 1,000,000 karena epsilon
# → Cap at 100 to remove extreme outliers

print("Fixing symmetry_ratio outliers...")

# Check distribution before fix
print(f"\nBefore fix:")
print(f"  symmetry_ratio_from > 100: "
      f"{(sales_clean['symmetry_ratio_from'] > 100).sum():,}")
print(f"  symmetry_ratio_to > 100  : "
      f"{(sales_clean['symmetry_ratio_to'] > 100).sum():,}")

# Cap at 100
CAP_VALUE = 100.0
sales_clean['symmetry_ratio_from'] = sales_clean[
    'symmetry_ratio_from'
].clip(upper=CAP_VALUE)

sales_clean['symmetry_ratio_to'] = sales_clean[
    'symmetry_ratio_to'
].clip(upper=CAP_VALUE)

print(f"\nAfter fix (capped at {CAP_VALUE}):")
print(f"  symmetry_ratio_from max: "
      f"{sales_clean['symmetry_ratio_from'].max():.2f}")
print(f"  symmetry_ratio_to max  : "
      f"{sales_clean['symmetry_ratio_to'].max():.2f}")

# Re-validate after fix
print(f"\n{'='*55}")
print(f"FEATURE VALIDATION AFTER FIX")
print(f"{'='*55}")

features_to_check = [
    'holding_time_hours',
    'pair_frequency',
    'time_since_last_hours',
    'symmetry_ratio_from',
    'symmetry_ratio_to'
]

wt     = sales_clean[sales_clean['is_wash_trading'] == 1]
normal = sales_clean[sales_clean['is_wash_trading'] == 0]

print(f"\n{'Feature':<25} {'Wash Trading':>15} "
      f"{'Normal':>15} {'Ratio':>10} {'Expected':>10}")
print("-" * 75)

expectations = {
    'holding_time_hours'    : '↓ lower',
    'pair_frequency'        : '↑ higher',
    'time_since_last_hours' : '↓ lower',
    'symmetry_ratio_from'   : '↑ higher',
    'symmetry_ratio_to'     : '↑ higher'
}

for feat in features_to_check:
    wt_mean  = wt[feat].mean()
    nor_mean = normal[feat].mean()
    ratio    = wt_mean / nor_mean if nor_mean != 0 else float('inf')
    expected = expectations[feat]
    print(f"{feat:<25} {wt_mean:>15.3f} "
          f"{nor_mean:>15.3f} {ratio:>10.2f}x {expected:>10}")

Fixing symmetry_ratio outliers...

Before fix:
  symmetry_ratio_from > 100: 7
  symmetry_ratio_to > 100  : 200,013

After fix (capped at 100.0):
  symmetry_ratio_from max: 100.00
  symmetry_ratio_to max  : 100.00

FEATURE VALIDATION AFTER FIX

Feature                      Wash Trading          Normal      Ratio   Expected
---------------------------------------------------------------------------
holding_time_hours                338.527         259.691       1.30x    ↓ lower
pair_frequency                      2.232           1.001       2.23x   ↑ higher
time_since_last_hours              48.754         131.188       0.37x    ↓ lower
symmetry_ratio_from                 0.919           0.694       1.32x   ↑ higher
symmetry_ratio_to                  11.625          15.631       0.74x   ↑ higher


In [14]:
# Drop symmetry_ratio_to — not aligned with expected behavior
# to_address wallet shows lower symmetry in wash trading
# which contradicts theoretical expectation
# symmetry_ratio_from already captures symmetry behavior sufficiently
sales_clean = sales_clean.drop(columns=['symmetry_ratio_to'])
print("Dropped symmetry_ratio_to ✅")

# Final feature list
features_final = [
    'transaction_value',       # Economic signal
    'holding_time_hours',      # Von Wachter et al. (2022)
    'pair_frequency',          # La Morgia et al. (2023)
    'time_since_last_hours',   # Liu et al. (2023)
    'symmetry_ratio_from',     # Von Wachter et al. (2022)
    'transfers_out_from',      # NFTGraph (2021)
    'transfers_in_from',       # NFTGraph (2021)
    'transfers_out_to',        # NFTGraph (2021)
    'transfers_in_to',         # NFTGraph (2021)
    'num_transitions',         # Ownership transitions
]

print(f"\nFinal features ({len(features_final)}):")
for f in features_final:
    print(f"  {f}")

Dropped symmetry_ratio_to ✅

Final features (10):
  transaction_value
  holding_time_hours
  pair_frequency
  time_since_last_hours
  symmetry_ratio_from
  transfers_out_from
  transfers_in_from
  transfers_out_to
  transfers_in_to
  num_transitions


In [16]:
output_path = '../dataset/nft_transfers_jun_aug_ready.csv'
sales_clean.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")
print(f"Shape: {sales_clean.shape}")


Saved: ../dataset/nft_transfers_jun_aug_ready.csv
Shape: (1563989, 21)
